# Предсказание стоимости недвижимости
Имеется 2 файла с данными - `train.csv` и `test.csv` для тренировки и тестирования моделей соответственно.

В файле `train.csv` есть поле `'price'`, а в `test.csv` нет.

Необходимо предсказать числовое поле `'price'` для данных в файле `test.csv`.

При помощи этой модели агрегаторы недвижимости смогут автоматически определять релевантную стоимость объекта и выбирать лучшие методики его продажи.

Итоговый файл должен иметь размерность 3087 строк на 2 столбца с полями `'id'` и `'price'`.

Минимальное значение метрики R^2: 0.6

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

Загрузим датасеты `train.csv` и `test.csv` и посмотрим на них

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900,3,1.00,1180,5650,1.0,N,0,...,7,1180.0,0.0,1955,0,98178,47.5112,-122.257,1340.0,5650.0
1,6414100192,20141209T000000,538000,3,2.25,2570,7242,2.0,N,0,...,7,2170.0,400.0,1951,1991,98125,47.7210,-122.319,1690.0,7639.0
2,5631500400,20150225T000000,180000,2,1.00,770,10000,1.0,N,0,...,6,770.0,0.0,1933,0,98028,47.7379,-122.233,2720.0,8062.0
3,2487200875,20141209T000000,604000,4,3.00,1960,5000,1.0,N,0,...,7,1050.0,910.0,1965,0,98136,47.5208,-122.393,1360.0,5000.0
4,1954400510,20150218T000000,510000,3,2.00,1680,8080,1.0,N,0,...,8,1680.0,0.0,1987,0,98074,47.6168,-122.045,1800.0,7503.0


In [4]:
train.describe(include='all')

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
count,1.834900e+04,18349,1.834900e+04,18349.000000,18349.000000,18349.000000,1.834900e+04,18349.000000,18349,18349.000000,...,18349.000000,18347.000000,18349.000000,18349.000000,18349.000000,18349.000000,18349.000000,18349.000000,18348.000000,18348.000000
unique,NaN,368,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,20140623T000000,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,118,NaN,NaN,NaN,NaN,NaN,NaN,18188,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,4.553878e+09,NaN,5.566407e+05,3.391247,2.142964,2129.272821,1.652298e+04,1.504115,NaN,0.254564,...,7.707940,1829.745844,299.552019,1971.328301,98.984577,98077.081748,47.561158,-122.209281,2022.170700,13792.803030
std,2.880970e+09,NaN,3.872222e+05,0.951008,0.784493,955.456759,4.472960e+04,0.538770,NaN,0.799973,...,1.209787,859.923209,454.778938,29.371158,433.344433,53.153357,0.138205,0.143748,708.405408,29447.919214
min,1.000102e+06,NaN,7.500000e+04,0.000000,0.000000,290.000000,5.200000e+02,1.000000,NaN,0.000000,...,1.000000,290.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000
25%,2.110200e+09,NaN,3.250000e+05,3.000000,1.750000,1440.000000,5.100000e+03,1.000000,NaN,0.000000,...,7.000000,1200.000000,0.000000,1952.000000,0.000000,98033.000000,47.473100,-122.325000,1500.000000,5150.000000
50%,3.885806e+09,NaN,4.600000e+05,3.000000,2.250000,1960.000000,7.757000e+03,1.500000,NaN,0.000000,...,7.000000,1590.000000,0.000000,1976.000000,0.000000,98065.000000,47.573800,-122.224000,1870.000000,7705.000000
75%,7.300201e+09,NaN,6.600000e+05,4.000000,2.500000,2630.000000,1.134000e+04,2.000000,NaN,0.000000,...,8.000000,2280.000000,580.000000,1997.000000,0.000000,98117.000000,47.678100,-122.118000,2420.000000,10458.000000


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 18349 entries, 0 to 18348
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             18349 non-null  int64  
 1   date           18349 non-null  str    
 2   price          18349 non-null  int64  
 3   bedrooms       18349 non-null  int64  
 4   bathrooms      18349 non-null  float64
 5   sqft_living    18349 non-null  int64  
 6   sqft_lot       18349 non-null  int64  
 7   floors         18349 non-null  float64
 8   waterfront     18349 non-null  str    
 9   view           18349 non-null  int64  
 10  condition      18349 non-null  str    
 11  grade          18349 non-null  int64  
 12  sqft_above     18347 non-null  float64
 13  sqft_basement  18349 non-null  float64
 14  yr_built       18349 non-null  int64  
 15  yr_renovated   18349 non-null  int64  
 16  zipcode        18349 non-null  int64  
 17  lat            18349 non-null  float64
 18  long           18

In [6]:
train.describe()

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,view,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
count,1.834900e+04,1.834900e+04,18349.000000,18349.000000,18349.000000,1.834900e+04,18349.000000,18349.000000,18349.000000,18347.000000,18349.000000,18349.000000,18349.000000,18349.000000,18349.000000,18349.000000,18348.000000,18348.000000
mean,4.553878e+09,5.566407e+05,3.391247,2.142964,2129.272821,1.652298e+04,1.504115,0.254564,7.707940,1829.745844,299.552019,1971.328301,98.984577,98077.081748,47.561158,-122.209281,2022.170700,13792.803030
std,2.880970e+09,3.872222e+05,0.951008,0.784493,955.456759,4.472960e+04,0.538770,0.799973,1.209787,859.923209,454.778938,29.371158,433.344433,53.153357,0.138205,0.143748,708.405408,29447.919214
min,1.000102e+06,7.500000e+04,0.000000,0.000000,290.000000,5.200000e+02,1.000000,0.000000,1.000000,290.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000
25%,2.110200e+09,3.250000e+05,3.000000,1.750000,1440.000000,5.100000e+03,1.000000,0.000000,7.000000,1200.000000,0.000000,1952.000000,0.000000,98033.000000,47.473100,-122.325000,1500.000000,5150.000000
50%,3.885806e+09,4.600000e+05,3.000000,2.250000,1960.000000,7.757000e+03,1.500000,0.000000,7.000000,1590.000000,0.000000,1976.000000,0.000000,98065.000000,47.573800,-122.224000,1870.000000,7705.000000
75%,7.300201e+09,6.600000e+05,4.000000,2.500000,2630.000000,1.134000e+04,2.000000,0.000000,8.000000,2280.000000,580.000000,1997.000000,0.000000,98117.000000,47.678100,-122.118000,2420.000000,10458.000000
max,9.842300e+09,7.700000e+06,33.000000,8.000000,13540.000000,1.651359e+06,3.500000,4.000000,13.000000,9410.000000,4820.000000,2015.000000,2015.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000


## Подготовим датасет к машинному обучению

Отделим сразу `X_train`, `y_train` в датасете `train`.

In [7]:
target = ['price']
X_train = train.copy().drop(target, axis=1)
y_train = train[target]
columns = X_train.columns

Напишем функцию по обработке датасета, чтобы применить его сразу и к train, и  у test. 

In [8]:
def transform(dataset):
    # Заменим {Y, N} на {1, 0}
    dataset['waterfront'] = dataset['waterfront'].apply(lambda x: 1 if x == 'Y' else 0)
    
    # Избавимся от колонки `id`, которая не влияет на прогноз
    dataset.drop(['id'], axis=1, inplace=True)

    # У `sqft_above` не хватает два значения, у `sqft_living15` и `sqft_lot1` по одному. 
    # Заменим их медианными значениями, чтобы не удалять.
    # Обработаем ВСЕ числовые значения таким образом.
    num_columns = dataset.select_dtypes(include=['int64', 'float64']).columns
    for col in num_columns:
        dataset[col] = dataset[col].fillna(dataset[col].median())
    
    # Пробежимся по всем числовым полям и "нормализуем" их исходя из межквартильного размаха
    for col in num_columns:
        Q1 = dataset[col].quantile(0.25)
        Q3 = dataset[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        dataset[col] = dataset[col].clip(lower, upper)
    

    # Если yr_renovated = 0, то заменим на yr_built
    dataset['yr_renovated'] = np.where(
        dataset['yr_renovated'] == 0,
        dataset['yr_built'],
        dataset['yr_renovated']
    )

    # Применим MinMaxScaler для yr_built и yr_renovated
    columns_to_scale = ['yr_built', 'yr_renovated']
    scaler = MinMaxScaler()
    dataset[columns_to_scale] = scaler.fit_transform(dataset[columns_to_scale])

    # Разберём date на date_year, date_month, date_day
    dataset['date_year'] = dataset['date'].str[:4].astype(int)
    dataset['date_month'] = dataset['date'].str[4:6].astype(int)
    dataset['date_day'] = dataset['date'].str[6:8].astype(int)
    dataset.drop(['date'], axis=1, inplace=True)

    # condition содержит значения ['Average', 'Very Good', 'Good', 'Poor', 'Fair']
    # сделаем mapping
    condition_mapping = {
        'Average': 3,
        'Very Good': 5,
        'Good': 4,
        'Poor': 1,
        'Fair': 2
        }
    dataset['condition'] = dataset['condition'].map(condition_mapping)


    

In [9]:
transform(X_train)

In [10]:
X_test = test.copy()
transform(X_test)

In [11]:
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance   # Посчитать важность признаков через перестановку
from sklearn.preprocessing import PolynomialFeatures    # Подсчёт полиномиальных признаков
from sklearn.pipeline import make_pipeline              # функция для создания пайплайна
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [12]:
degree = 3
model = make_pipeline(PolynomialFeatures(degree), Lasso())
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('polynomialfeatures', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",3
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False


In [13]:
y_pred = model.predict(X_test)

In [14]:
result = pd.DataFrame(y_pred, columns=['price'])
result = pd.concat([test['id'], result.astype('float64')], axis=1)

In [15]:
result

,id,price
0,1917300105,150202.946475
1,2781280300,361432.129683
2,2310110230,341820.075770
3,8944300110,211197.795308
4,4232401265,799061.853195
...,...,...
3082,3066400080,769496.904135
3083,6821101731,424253.452012
3084,7202340720,498993.186928
3085,4315700505,614316.271210


In [16]:
result.to_csv('result.csv', index=False)

**Результат:** R^2 = 0.6863